# Local Danish ASR + Diarization

Replacement for the `insanely-fast-whisper` notebook. Runs fully **local** on a single GPU (~23 GB VRAM):

- **ffmpeg** → 16 kHz mono WAV
- **faster-whisper** (`large-v3`, fp16) → Danish transcription with word timestamps
- **pyannote.audio** (`speaker-diarization-3.1`) → who-spoke-when
- Word-to-speaker alignment → JSON in the **same schema** as the old pipeline
- **python-docx** → `*_edit.docx` (same layout as before)

No cloud APIs, no containers, no benchmarking. Audio never leaves the machine.

**One-time setup:** accept the user agreements for
[pyannote/speaker-diarization-3.1](https://huggingface.co/pyannote/speaker-diarization-3.1) and
[pyannote/segmentation-3.0](https://hf.co/pyannote/segmentation-3.0), then create a HF token at
[huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).


## 1. Install dependencies

**Machine-independent and Jupyter-web-safe — nothing CUDA-specific is pinned.**

This notebook may be run from VS Code, JupyterLab, or the classic Jupyter web UI, where
the working directory can be the repo root, the `notebooks/` folder, or another parent.
The setup cell therefore searches for `transcribe-faster-whisper.requirements.in` next to
this notebook, and you can override that search with `TRANSCRIBE_FAST_WHISPER_DIR` if a
Jupyter deployment starts somewhere unusual.

The two environment problems this cell protects against are:

1. `torch` ↔ `torchaudio` CUDA-build mismatch: pyannote imports `torchaudio` at load time,
   and e.g. torch `cu130` with torchaudio `cu128` dies with a hard `libcudart.so.*` /
   `libc10.so` error. The cell first tries to import `torchaudio`; if it loads, it touches
   nothing. Only if import fails does it reinstall `torchaudio` from the wheel index that
   matches **torch's own build**, derived from `torch.version.cuda` (not `nvidia-smi`).
2. Missing CUDA library search path inside Jupyter: faster-whisper/CTranslate2 may fail
   with `libcublas.so.12 is not found` even when the NVIDIA pip packages are installed.
   The cell discovers NVIDIA library folders from the active kernel's `site-packages`,
   updates `LD_LIBRARY_PATH`, and preloads the key CUDA `.so` files in-process.

The remaining pure-Python deps are installed from `transcribe-faster-whisper.requirements.in`.
`torch`/`torchaudio` are deliberately **not** listed there. If torchaudio is reinstalled,
**restart the kernel** and run this cell again. `ffmpeg` must be on `PATH`
(`sudo apt install ffmpeg`).


In [ ]:
import ctypes
import os
import site
import subprocess
import sys
from pathlib import Path

# This notebook's pure-Python deps live next to it in the repo. torch/torchaudio are
# NOT listed there — torchaudio is reconciled to the machine's torch build (see below).
NOTEBOOK_NAME = "transcribe-faster-whisper"
REQ_FILENAME = f"{NOTEBOOK_NAME}.requirements.in"

DEFAULT_REQUIREMENTS = """\
# Top-level Python dependencies for transcribe-faster-whisper.ipynb
# torch / torchaudio are reconciled automatically to the machine's CUDA build.
speechbrain>=1.0
pyannote.audio>=3.3
faster-whisper
python-docx
tqdm
numpy
"""


def unique_existing(paths):
    seen = set()
    out = []
    for path in paths:
        try:
            resolved = Path(path).expanduser().resolve()
        except Exception:
            continue
        if resolved.exists() and resolved not in seen:
            seen.add(resolved)
            out.append(resolved)
    return out


def locate_requirements_dir():
    """Find the repo-side requirements file from Jupyter web or VS Code.

    Plain notebooks do not expose their file path to Python reliably. This supports
    the common launch points: the notebook folder, the repo root, or a parent folder.
    Set TRANSCRIBE_FAST_WHISPER_DIR to override in unusual Jupyter deployments.
    """
    override = os.environ.get("TRANSCRIBE_FAST_WHISPER_DIR")
    cwd = Path.cwd()
    candidates = []
    if override:
        candidates.append(Path(override))
    for root in [cwd, *cwd.parents]:
        candidates.extend([root, root / "notebooks"])
    for folder in unique_existing(candidates):
        if (folder / REQ_FILENAME).exists() or (folder / f"{NOTEBOOK_NAME}.ipynb").exists():
            return folder
    fallback = cwd / "notebooks" if (cwd / "notebooks").exists() else cwd
    return fallback.resolve()


REQ_DIR = locate_requirements_dir()
REQ_IN = str(REQ_DIR / REQ_FILENAME)


def pip_install(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)


def python_site_roots():
    roots = []
    try:
        roots.extend(site.getsitepackages())
    except Exception:
        pass
    try:
        roots.append(site.getusersitepackages())
    except Exception:
        pass
    roots.extend(p for p in sys.path if "site-packages" in str(p))
    return unique_existing(roots)


def nvidia_cuda_lib_dirs():
    dirs = []
    for root in python_site_roots():
        nvidia = root / "nvidia"
        if nvidia.exists():
            dirs.extend(p for p in nvidia.rglob("lib") if any(p.glob("*.so*")))
    return unique_existing(dirs)


def preload_shared_libraries(lib_dirs):
    """Load NVIDIA pip-package CUDA libs into the current Jupyter process.

    Updating LD_LIBRARY_PATH helps subprocesses, but some libraries are loaded inside
    this already-running Python process. Preloading by absolute path makes ctranslate2
    and torchaudio able to resolve libcublas/libcudart without relying on shell setup.
    """
    wanted = [
        "libcudart.so*",
        "libnvrtc.so*",
        "libcublasLt.so*",
        "libcublas.so*",
        "libcudnn.so*",
        "libcudnn_*.so*",
    ]
    loaded = []
    for pattern in wanted:
        for lib_dir in lib_dirs:
            for lib in sorted(lib_dir.glob(pattern)):
                if lib.is_file():
                    try:
                        ctypes.CDLL(str(lib), mode=ctypes.RTLD_GLOBAL)
                        loaded.append(lib.name)
                    except OSError:
                        pass
    return loaded


def bootstrap_cuda_library_path():
    lib_dirs = nvidia_cuda_lib_dirs()
    if not lib_dirs:
        print("No NVIDIA pip CUDA library directories found in this kernel.")
        return []

    existing = [p for p in os.environ.get("LD_LIBRARY_PATH", "").split(os.pathsep) if p]
    lib_dir_strings = [str(p) for p in lib_dirs]
    os.environ["LD_LIBRARY_PATH"] = os.pathsep.join(lib_dir_strings + [p for p in existing if p not in lib_dir_strings])
    loaded = preload_shared_libraries(lib_dirs)
    print(f"Registered {len(lib_dirs)} NVIDIA CUDA library folder(s) for this kernel.")
    if loaded:
        print("Preloaded CUDA libs:", ", ".join(sorted(set(loaded))[:8]))
    return lib_dirs


CUDA_LIB_DIRS = bootstrap_cuda_library_path()


def torchaudio_imports():
    """True iff torchaudio's compiled extension loads. This is the ONLY thing that
    matters for pyannote — it imports torchaudio at load time, and a torch/torchaudio
    CUDA-build mismatch makes that import die with a libcudart/libc10 OSError."""
    try:
        import torchaudio  # noqa: F401
        return True
    except Exception:
        return False


def torch_build_tag():
    """CUDA wheel tag torch was actually built against, e.g. 'cu130' / 'cu128' / 'cpu'.
    Source of truth is torch.version.cuda (NOT nvidia-smi: the driver reports its max
    CUDA, e.g. 13.2, for which no PyTorch wheel index exists)."""
    try:
        import torch
    except Exception:
        return None
    cu = getattr(torch.version, "cuda", None)
    if not cu:
        return "cpu"
    major, minor = (cu.split(".") + ["0"])[:2]
    return f"cu{major}{minor}"


# 1) Reconcile torchaudio ONLY if it actually fails to import (the failure that bit us
#    before). When it imports fine we touch nothing — fully machine-independent.
if torchaudio_imports():
    print("torchaudio imports cleanly — no reconcile needed.")
else:
    tag = torch_build_tag()
    if tag is None:
        print("torch is not installed yet — skipping; pyannote will pull a torchaudio.")
    else:
        index = f"https://download.pytorch.org/whl/{tag}"
        print(f"torchaudio failed to import; reinstalling to match torch build '{tag}' via {index}")
        pip_install("--force-reinstall", "--no-deps", "--index-url", index, "torchaudio")
        print("torchaudio reconciled — RESTART THE KERNEL, then run this cell again.")

# 2) Remaining pure-Python deps (machine-independent, unpinned).
if not os.path.exists(REQ_IN):
    with open(REQ_IN, "w", encoding="utf-8") as f:
        f.write(DEFAULT_REQUIREMENTS)
    print("Created:", REQ_IN)
print("Installing project dependencies from:", REQ_IN)
pip_install("-r", REQ_IN)

# Re-scan in case installing dependencies added NVIDIA pip packages.
CUDA_LIB_DIRS = bootstrap_cuda_library_path()

print("Dependencies ready.")
# ffmpeg must be on PATH (apt: `sudo apt install ffmpeg`)
print(subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True).stdout.splitlines()[0])


## 2. Configure

Edit the three values below, then run the rest top-to-bottom.

In [ ]:
import getpass
import glob
import os

# --- edit these ---
PATH_AUDIO = "/work/speech/interviews"          # folder with the interview files
EXTENSIONS = ["mp3", "wma", "WMA", "m4a", "wav", "flac", "ogg", "mp4", "MP3"]
NUM_SPEAKERS = 2                              # set to None to auto-detect
# ------------------

WHISPER_MODEL = "large-v3"
LANGUAGE = "da"
DEVICE = "cuda"
COMPUTE_TYPE = "float16"                      # fits comfortably in 23 GB
DIARIZATION_MODEL = "pyannote/speaker-diarization-3.1"

if not os.environ.get("HUGGINGFACE_AUTH_TOKEN"):
    os.environ["HUGGINGFACE_AUTH_TOKEN"] = getpass.getpass("HUGGINGFACE_AUTH_TOKEN: ")
HF_TOKEN = os.environ["HUGGINGFACE_AUTH_TOKEN"]

files = []
for ext in EXTENSIONS:
    files.extend(glob.glob(os.path.join(PATH_AUDIO, f"*.{ext}")))
files = sorted(set(files))
print(f"{len(files)} file(s) to transcribe:")
for f in files:
    print(" ", f)

## 3. Load models (once)

In [ ]:
import warnings

# Re-apply CUDA library discovery before importing model libraries. This matters when
# the notebook is run from Jupyter web, where the shell's LD_LIBRARY_PATH may not include
# NVIDIA pip-package libraries from the active kernel environment.
if "bootstrap_cuda_library_path" in globals():
    CUDA_LIB_DIRS = bootstrap_cuda_library_path()

# We decode audio ourselves (ffmpeg + stdlib wave) and hand pyannote an in-memory
# waveform, so pyannote NEVER calls torchcodec. The warning torchcodec emits at import
# time (e.g. "libtorchcodec ... libnvrtc.so.13: cannot open shared object file") is
# therefore harmless noise — silence it for clean output.
warnings.filterwarnings("ignore", message=".*torchcodec.*")

import torch
from faster_whisper import WhisperModel
from pyannote.audio import Pipeline

print("Loading faster-whisper:", WHISPER_MODEL)
asr = WhisperModel(WHISPER_MODEL, device=DEVICE, compute_type=COMPUTE_TYPE)

print("Loading diarization pipeline:", DIARIZATION_MODEL)
diarizer = Pipeline.from_pretrained(DIARIZATION_MODEL, token=HF_TOKEN)
diarizer.to(torch.device(DEVICE))
print("Models ready.")


## 4. Pipeline functions

`transcribe_and_diarize` writes a JSON file next to each audio file using the **same schema** the old
`insanely-fast-whisper` notebook produced, so the existing docx-export step keeps working unchanged:

```json
{"speakers": [{"timestamp": [start, end], "speaker": "SPEAKER_00", "text": "..."}, ...]}
```

In [ ]:
import json
import shutil
import subprocess
import tempfile
import wave
from bisect import bisect_right
from pathlib import Path

import numpy as np


def to_wav_16k_mono(src: str, dst: str) -> None:
    """Convert any audio to 16 kHz mono 16-bit PCM WAV using ffmpeg.

    Forcing pcm_s16le makes the output decodable by Python's stdlib ``wave``
    module, so we never depend on torchaudio/torchcodec backends for reading.
    """
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error",
         "-i", src, "-ac", "1", "-ar", "16000", "-vn",
         "-c:a", "pcm_s16le", "-f", "wav", dst],
        check=True,
    )


def load_wav_in_memory(wav_path: str) -> dict:
    """Decode a 16 kHz mono 16-bit PCM WAV into an in-memory waveform dict.

    Returns ``{'waveform': (1, time) float32 tensor, 'sample_rate': int}`` which
    pyannote.audio accepts directly. This bypasses torchcodec/torchaudio entirely,
    so the pipeline keeps working across PyTorch/ffmpeg/torchcodec version churn.
    """
    with wave.open(wav_path, "rb") as wf:
        sr = wf.getframerate()
        n_channels = wf.getnchannels()
        sampwidth = wf.getsampwidth()
        raw = wf.readframes(wf.getnframes())

    if sampwidth != 2:
        raise ValueError(f"expected 16-bit PCM, got {sampwidth * 8}-bit")

    data = np.frombuffer(raw, dtype="<i2").astype(np.float32) / 32768.0
    if n_channels > 1:
        data = data.reshape(-1, n_channels).mean(axis=1)
    waveform = torch.from_numpy(np.ascontiguousarray(data)).unsqueeze(0)  # (1, time)
    return {"waveform": waveform, "sample_rate": sr}


def run_whisper(wav_path: str):
    """Return list of {'start','end','text'} word dicts from faster-whisper."""
    segments, _ = asr.transcribe(
        wav_path,
        language=LANGUAGE,
        task="transcribe",
        vad_filter=True,
        word_timestamps=True,
        beam_size=5,
    )
    words = []
    for seg in segments:
        if not seg.words:
            words.append({"start": float(seg.start), "end": float(seg.end),
                          "text": seg.text.strip()})
            continue
        for w in seg.words:
            if w.start is None or w.end is None:
                continue
            words.append({"start": float(w.start), "end": float(w.end),
                          "text": w.word})
    return words


def run_diarization(wav_path: str):
    """Return sorted list of (start, end, speaker)."""
    kwargs = {}
    if NUM_SPEAKERS:
        kwargs["num_speakers"] = NUM_SPEAKERS
    # Feed an in-memory waveform so pyannote never invokes torchcodec file decoding.
    audio = load_wav_in_memory(wav_path)
    result = diarizer(audio, **kwargs)
    # pyannote.audio 3.3+ returns a DiarizeOutput wrapper; older versions return Annotation directly
    annotation = getattr(result, "speaker_diarization", result)
    turns = [(float(t.start), float(t.end), str(spk))
             for t, _, spk in annotation.itertracks(yield_label=True)]
    turns.sort()
    return turns


def assign_speakers(words, turns):
    """Tag each word with the speaker whose turn covers its midpoint
    (falls back to the nearest turn). Then merge consecutive same-speaker words."""
    if not turns:
        return [{"timestamp": [w["start"], w["end"]],
                 "speaker": "SPEAKER_00",
                 "text": w["text"]} for w in words]

    starts = [t[0] for t in turns]
    tagged = []
    for w in words:
        mid = 0.5 * (w["start"] + w["end"])
        idx = bisect_right(starts, mid) - 1
        candidates = []
        if 0 <= idx < len(turns):
            candidates.append(turns[idx])
        if idx + 1 < len(turns):
            candidates.append(turns[idx + 1])
        if not candidates:
            candidates = [turns[0]]
        # prefer a turn that actually contains the midpoint, else nearest
        containing = [t for t in candidates if t[0] <= mid <= t[1]]
        if containing:
            spk = containing[0][2]
        else:
            spk = min(candidates, key=lambda t: min(abs(mid - t[0]), abs(mid - t[1])))[2]
        tagged.append((w, spk))

    # merge consecutive words with the same speaker into utterances
    utterances = []
    for w, spk in tagged:
        if utterances and utterances[-1]["speaker"] == spk:
            utterances[-1]["timestamp"][1] = w["end"]
            utterances[-1]["text"] += w["text"] if w["text"].startswith(" ") else " " + w["text"]
        else:
            utterances.append({
                "timestamp": [w["start"], w["end"]],
                "speaker": spk,
                "text": w["text"],
            })
    for u in utterances:
        u["text"] = u["text"].strip()
    return utterances


def transcribe_and_diarize(audio_path: str) -> str:
    """Full pipeline for one file. Writes <name>-transcription.json next to the audio."""
    audio_path = str(audio_path)
    out_json = os.path.splitext(audio_path)[0] + "-transcription.json"

    with tempfile.TemporaryDirectory() as tmp:
        wav = os.path.join(tmp, "audio.wav")
        to_wav_16k_mono(audio_path, wav)
        words = run_whisper(wav)
        turns = run_diarization(wav)

    speakers = assign_speakers(words, turns)
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump({"speakers": speakers}, f, ensure_ascii=False, indent=2)
    return out_json


assert shutil.which("ffmpeg"), "ffmpeg not found on PATH"


## 5. Smoke test

Runs the full pipeline on a 3-second generated tone to confirm the in-memory audio
bypass, faster-whisper, and pyannote all work in this environment before touching
real files.


In [ ]:
import tempfile


def smoke_test():
    with tempfile.TemporaryDirectory() as tmp:
        raw = os.path.join(tmp, "tone.wav")
        # 3 s, 16 kHz mono sine via ffmpeg (also exercises the ffmpeg path)
        subprocess.run(
            ["ffmpeg", "-y", "-loglevel", "error", "-f", "lavfi",
             "-i", "sine=frequency=220:duration=3", "-ac", "1", "-ar", "16000",
             "-c:a", "pcm_s16le", "-f", "wav", raw],
            check=True,
        )
        wav = os.path.join(tmp, "audio.wav")
        to_wav_16k_mono(raw, wav)
        audio = load_wav_in_memory(wav)
        assert audio["sample_rate"] == 16000
        assert audio["waveform"].shape[0] == 1 and audio["waveform"].shape[1] > 0
        _ = run_whisper(wav)          # ASR runs
        _ = run_diarization(wav)      # pyannote accepts in-memory waveform
    print("Smoke test passed: decode → ASR → diarization all work.")


smoke_test()
SMOKE_TEST_PASSED = True


## 6. Batch transcribe

In [ ]:
from tqdm.auto import tqdm

failed = []
for idx, audio in enumerate(tqdm(files, desc="files"), start=1):
    print(f"\n[{idx}/{len(files)}] {audio}")
    try:
        out = transcribe_and_diarize(audio)
        print("  →", out)
    except Exception as e:
        print(f"  !! failed: {e}")
        failed.append((audio, str(e)))

if failed:
    print("\nFailed files:")
    for a, e in failed:
        print(" ", a, "->", e)
else:
    print("\nAll files transcribed.")

## 6. Convert transcriptions to Word

Identical to the old notebook — same `*_edit.docx` output, same layout.

In [ ]:
import glob
import json
import os

from docx import Document
from docx.enum.text import WD_ALIGN_PARAGRAPH


def diarization_to_docx_edit(audio_file_diarization):
    audio_file_basename = os.path.basename(audio_file_diarization)

    doc = Document()
    paragraph = doc.add_paragraph()
    paragraph.add_run(audio_file_basename).bold = True
    paragraph.paragraph_format.alignment = WD_ALIGN_PARAGRAPH.CENTER

    with open(audio_file_diarization, "r", encoding="utf-8") as f:
        data = json.load(f)

    for entry in data["speakers"]:
        paragraph = doc.add_paragraph()
        paragraph.add_run(str(entry["timestamp"]) + "\n")
        paragraph.add_run(str(entry["speaker"]) + "\n")
        paragraph.add_run(str(entry["text"]).strip())

    docx_filename = audio_file_diarization.replace(".json", "_edit.docx")
    doc.save(docx_filename)


files_json = sorted(glob.glob(os.path.join(PATH_AUDIO, "*-transcription.json")))
print(f"{len(files_json)} JSON file(s) → docx")
for idx, file in enumerate(files_json, start=1):
    print(f"  [{idx}/{len(files_json)}] {file}")
    diarization_to_docx_edit(file)
print("Done!")